# 02 · Gold dataset review

Reviewer annotations (`datasets/annotations/<reviewer>.jsonl`), disagreements, missing evidence references, adjudication progress and finalized gold cases.

**Read-only:** this notebook never writes labels. Adjudications are recorded with `engine.datasets.append_adjudication`, which refuses to overwrite an existing decision. The last cell proves no dataset file changed.

In [ ]:
import os, sys, json
from dataclasses import asdict
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_colwidth", 120); pd.set_option("display.width", 200)
print("research root:", ROOT)

In [ ]:
import glob
from engine.datasets import (DATASET_DIR, file_sha256, load_split, load_annotations, load_adjudications,
                             gold_status, finalized_gold, cohen_kappa, FINAL_STATUSES)
from engine.notebook import banner
from engine import synthetic
before = {p: file_sha256(p) for p in glob.glob(os.path.join(DATASET_DIR, "**", "*.jsonl"), recursive=True)}
claims = load_split("development") + load_split("gold")
annotations, adjudications = load_annotations(), load_adjudications()
USE_SYNTHETIC = not annotations
if USE_SYNTHETIC:
    banner("No real annotations exist yet. The tables below run on SYNTHETIC claims and reviewers (ids SYN-*) "
           "only to show the review workflow works. They are not gold labels.")
    claims = synthetic.synthetic_claims()
    annotations, adjudications = synthetic.synthetic_annotations(claims)
print(len(claims), "claims |", len(annotations), "annotations |", len(adjudications), "adjudications")

## Reviewer annotations

In [ ]:
ann = pd.DataFrame([{**asdict(a), "n_supporting": len(a.supporting_evidence), "n_contradictory": len(a.contradictory_evidence)}
                    for a in annotations])
display(ann[["claim_id", "reviewer_id", "gold_label", "n_supporting", "n_contradictory", "review_notes"]].head(40))
pd.crosstab(ann.reviewer_id, ann.gold_label, margins=True)

## Adjudication progress

In [ ]:
status = pd.DataFrame(gold_status(claims, annotations, adjudications))
progress = status.adjudication_status.value_counts().rename("claims").to_frame()
progress["share"] = (progress.claims / len(status)).round(2)
display(progress)
fig, ax = plt.subplots(figsize=(6, 2.6))
progress.claims.plot.barh(ax=ax, title="Review state per claim"); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

## Reviewer disagreements

In [ ]:
dis = status[status.adjudication_status.isin(["DISAGREED"]) |
             ((status.adjudication_status == "ADJUDICATED") & status.reviewer_labels.apply(lambda d: len(set(d.values())) > 1))]
dis[["claim_id", "claim_category", "reviewer_labels", "adjudication_status", "final_label"]]

## Inter-annotator agreement (Cohen's κ per reviewer pair)

In [ ]:
from itertools import combinations
by = {}
for a in annotations:
    by.setdefault(a.reviewer_id, {})[a.claim_id] = a.gold_label
rows = []
for r1, r2 in combinations(sorted(by), 2):
    shared = sorted(set(by[r1]) & set(by[r2]))
    pairs = [(by[r1][c], by[r2][c]) for c in shared]
    rows.append({"pair": f"{r1} vs {r2}", "shared_claims": len(shared), "raw_agreement": (sum(a == b for a, b in pairs) / len(pairs)) if pairs else None,
                 "cohen_kappa": cohen_kappa(pairs)})
pd.DataFrame(rows)

## Missing evidence references

A SUPPORTED / CONTRADICTED annotation must point at the evidence (filing accession + verbatim quote, or XBRL concept + period).

In [ ]:
miss = status[status.reviewers_missing_evidence.apply(len) > 0][["claim_id", "reviewer_labels", "reviewers_missing_evidence"]]
print(len(miss), "claim(s) with an annotation lacking evidence references")
miss

## Finalized gold cases

In [ ]:
final = finalized_gold(claims, annotations, adjudications)
fin = pd.DataFrame([{"claim_id": k, "label": v["label"], "status": v["status"],
                     "supporting_refs": len(v["supporting_evidence"]), "contradictory_refs": len(v["contradictory_evidence"])}
                    for k, v in final.items()])
print(f"{len(fin)} finalized of {len(status)} claims ({', '.join(FINAL_STATUSES)})")
fin

In [ ]:
after = {p: file_sha256(p) for p in glob.glob(os.path.join(DATASET_DIR, "**", "*.jsonl"), recursive=True)}
assert after == before, "a dataset file changed while this notebook ran"
print("OK: no dataset file was modified by this notebook")